# Spec-Craft Quickstart: Spec-Driven Development with AI & Obsidian

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akuroiwa/spec-craft/blob/main/docs/source/notebooks/spec_craft_quickstart.ipynb)

Welcome to the **Spec-Craft** interactive tutorial on Google Colab!

**Spec-Craft** is an Obsidian-based Spec-Driven Development (SDD) toolkit designed for AI agents and developers. It establishes a clean separation of concerns:
- **Strategy (Obsidian)**: Defines the *What* and *Why* (the project's soul).
- **Tactics (spec-kit & code)**: Executes the *How* (iterative implementation cycles).

In this notebook, you will learn how to:
1. Set up a local LLM environment using **Ollama** (with Gemma 4 QAT) and **sigoden/aichat** with `uv`.
2. Generate an Obsidian strategic feature specification (`obsidian/strategy_task_tracker.md`).
3. Inspect and parse the strategic roadmap using **Spec-Craft**'s `StoryboardParser`.
4. Automatically bridge strategy into tactical implementation (generating Python code and `pytest` test suites).
5. Verify behavioral correctness by running tests directly in Colab.

In [ ]:
# ==============================================================================
# 1. Environment Setup (Rust toolchain, aichat, Ollama, Gemma 4 QAT, and spec-craft)
# ==============================================================================
import os, time, subprocess

print("📦 1. Installing system dependencies...")
!apt-get update -qq && apt-get install -y -qq curl zstd git build-essential > /dev/null

print("📥 2. Installing sigoden/aichat CLI...")
# Download official prebuilt binary for fastest setup
!curl -sSF https://raw.githubusercontent.com/sigoden/aichat/main/scripts/install.sh | sh -s -- --to /usr/local/bin

print("🚀 3. Installing and launching Ollama background service...")
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama in the background
subprocess.Popen(["ollama", "serve"])

# Wait until Ollama server is ready (health check)
for _ in range(10):
    time.sleep(1)
    try:
        res = subprocess.run(["curl", "-s", "http://127.0.0.1:11434"], capture_output=True, text=True)
        if "Ollama is running" in res.stdout:
            print("   Ollama server is active!")
            break
    except Exception:
        pass

print("🧠 4. Pulling Gemma 4 QAT model & creating alias...")
# --- Model Setup: Gemma 4 QAT (Quantization-Aware Training) ---
# [Recommended Default for Colab T4 / GPU] Gemma 4 12B QAT (~7.2 GB VRAM)
!ollama pull hf.co/google/gemma-4-12B-it-qat-q4_0-gguf:Q4_0
!ollama cp hf.co/google/gemma-4-12B-it-qat-q4_0-gguf:Q4_0 gemma4-12b-qat

# --- Alternative Models (Uncomment as needed) ---
# [Ultra-Lightweight / Edge] Gemma 4 E2B QAT (~2.9 GB VRAM):
# !ollama pull hf.co/google/gemma-4-E2B-it-qat-q4_0-gguf:Q4_0 && ollama cp hf.co/google/gemma-4-E2B-it-qat-q4_0-gguf:Q4_0 gemma4-e2b-qat

# [Fast Standard Fallback] Qwen 2.5 1.5B (~1.5 GB VRAM):
# !ollama pull qwen2.5:1.5b

print("🛠️ 5. Installing Python dependencies & spec-craft...")
!pip install -q uv pytest
!uv pip install --system git+https://github.com/akuroiwa/spec-craft.git

print("✅ Core environment ready!")

In [ ]:
# ==============================================================================
# 2. Dynamic aichat Configuration (Auto-detect available Ollama models)
# ==============================================================================
import os, subprocess, glob

# 1. Clean previous cache and config
config_dir = os.path.expanduser("~/.config/aichat")
if os.path.exists(config_dir):
    for f in glob.glob(f"{config_dir}/*"):
        try:
            os.remove(f)
        except Exception:
            pass

# 2. Detect pulled models from Ollama
try:
    res = subprocess.run(["ollama", "list"], capture_output=True, text=True)
    lines = res.stdout.strip().split("\n")[1:]
    pulled_models = [line.split()[0] for line in lines if line]
except Exception:
    pulled_models = ["gemma4-12b-qat:latest"]

# Select default model
if "gemma4-12b-qat:latest" in pulled_models:
    default_model = "gemma4-12b-qat:latest"
elif "gemma4-12b-qat" in pulled_models:
    default_model = "gemma4-12b-qat"
elif "gemma4-e2b-qat:latest" in pulled_models:
    default_model = "gemma4-e2b-qat:latest"
else:
    default_model = pulled_models[0] if pulled_models else "gemma4-12b-qat:latest"

# 3. Build models list for config.yaml (OpenAI-compatible endpoint)
models_list = "\n".join([f"      - name: {m}" for m in pulled_models])

aichat_config = f"""
model: ollama:{default_model}

clients:
  - type: openai-compatible
    name: ollama
    api_base: http://127.0.0.1:11434/v1
    models:
{models_list}
"""

os.makedirs(config_dir, exist_ok=True)
with open(os.path.join(config_dir, "config.yaml"), "w") as f:
    f.write(aichat_config.strip())

print(f"✅ aichat configured with active model: ollama:{default_model}")
print(f"   Detected models: {pulled_models}")

> **Note (Troubleshooting): Emergency Ollama Server Restart**
> If the Ollama server ever becomes unresponsive, execute the cell below to safely restart it.

In [ ]:
# Emergency restart for Ollama service
!pkill -9 -f ollama || true
!ollama serve > /dev/null 2>&1 &
!sleep 2
!curl -s http://localhost:11434

## 1. Verify AI Chat & spec-craft Prerequisites

Check the environment readiness using `spec-craft check-env` and verify `aichat` non-streaming query execution (`-S` flag).

In [ ]:
# Verify environment readiness
!spec-craft check-env

# Test aichat query with non-streaming flag (-S)
!aichat -S "Hello! Confirm in one short sentence that you are ready for Spec-Driven Development."

## 2. Step 1: Draft the Strategic Specification in Obsidian (`obsidian/`)

In Spec-Craft, development begins with a strategic vision in Obsidian. We define the **What** and **Why** before diving into implementation details.

Let's prompt `aichat` to draft a feature specification for a **Markdown Task Progress Analyzer CLI tool** (`obsidian/strategy_task_tracker.md`).

In [ ]:
import os
os.makedirs("obsidian", exist_ok=True)

# Draft Obsidian specification with frontmatter and structured sections
!aichat -S "Create an Obsidian-compatible feature roadmap Markdown note with YAML frontmatter. \
Title: 'Markdown Task Tracker CLI'. \
Include YAML frontmatter with title, tags: [sdd, cli, utility], and status: draft. \
Include H1 sections: '# 1. Overview', '# 2. User Stories', '# 3. Technical Requirements'. \
The tool parses a markdown string or file, finds all '- [ ]' (pending) and '- [x]' (completed) checklist items, and calculates total, completed, pending, and completion percentage." > obsidian/strategy_task_tracker.md

# Display the generated Obsidian strategic specification
!cat obsidian/strategy_task_tracker.md

## 3. Step 2: Inspect and Parse the Strategic Roadmap with Spec-Craft

Spec-Craft provides structured parsers and MCP tools to inspect Obsidian storyboards, extracting metadata and sections safely without losing context.

In [ ]:
from pathlib import Path
from spec_craft.core.parser import StoryboardParser

# Parse the generated storyboard note
parser = StoryboardParser(Path("obsidian"))
storyboard = parser.parse("strategy_task_tracker.md")

print(f"📄 File Path: {storyboard.file_path}")
print(f"🏷️  Metadata: {storyboard.metadata}")
print(f"📑 Sections Found ({len(storyboard.sections)}):")
for section in storyboard.sections:
    print(f"   [Level {section.level}] {section.title}")

## 4. Step 3: Automated Tactical Implementation (spec-kit Workflow)

Following Spec-Craft's SDD lifecycle:
1. `spec-craft` guides the tactical phase (translating Strategy into executable Python code and tests).
2. We generate the implementation module (`task_tracker.py`) and unit tests (`test_task_tracker.py`).

In [ ]:
# Generate the Python implementation based on the Obsidian spec
!aichat -S -f obsidian/strategy_task_tracker.md "Implement the complete Python module 'task_tracker.py'. \
Provide a class 'TaskTracker' and a function 'analyze_tasks(content: str) -> dict' \
returning {'total': int, 'completed': int, 'pending': int, 'percentage': float}. \
Output ONLY raw Python code without Markdown backticks." > task_tracker.py

# Generate unit tests with pytest
!aichat -S -f task_tracker.py "Write comprehensive pytest unit tests in 'test_task_tracker.py'. \
Test empty text, all pending, all completed, and mixed tasks. Output ONLY raw Python code without Markdown backticks." > test_task_tracker.py

print("✨ Generated task_tracker.py and test_task_tracker.py")

## 5. Step 4: Verification and Test Execution

Run `pytest` to verify that the generated implementation passes all acceptance criteria.

In [ ]:
# Execute unit tests
!pytest test_task_tracker.py -v

## Summary & Next Steps

Congratulations! You have completed the full Spec-Driven Development cycle in Google Colab:
- **Strategy (Obsidian)**: Captured vision and user requirements.
- **Spec-Craft & aichat**: Parsed structure and bridged strategy to tactics.
- **Tactics (spec-kit & pytest)**: Generated clean, test-verified implementation.

### Resources
- **GitHub Repository**: [https://github.com/akuroiwa/spec-craft](https://github.com/akuroiwa/spec-craft)
- **Documentation**: [https://spec-craft.readthedocs.io/](https://spec-craft.readthedocs.io/)
- **Zenn Book / Articles**: [spec-craft Development Notes](https://zenn.dev/akuroiwa/books/dev-note-spec-craft)